# Stage 1C prototype: local semantic-parameter codebook training

This notebook creates a **demo-only** semantic parameter codebook from separate local images. It uses CLIP only as an offline teacher. Runtime codebook inference consumes SwinJSCC semantic representations, never CLIP embeddings.

It is not a research-quality training split and must not use `Test/images`.

## 1. Setup and reproducibility

In [ ]:
from __future__ import annotations

import json
import os
import platform
import subprocess
import sys
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from the SemanticSchedulerEnd2End repository or a child directory.')


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
SOURCE_ROOT = PROJECT_ROOT / 'src'
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))


def git_commit() -> str:
    try:
        return subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    except Exception:
        return 'unavailable'


RUN_CONTEXT = {
    'project_root': str(PROJECT_ROOT),
    'git_commit': git_commit(),
    'python': platform.python_version(),
}
print(json.dumps(RUN_CONTEXT, indent=2))


## 2. Configuration and strict input separation

In [ ]:
from utils.config_loader import load_yaml_config
from EncDecPipeline.Preprocessing.image_loader import discover_images, file_sha256

CONFIG_PATH = PROJECT_ROOT / 'config' / 'stage1c_parameter_codebook.yaml'
config = load_yaml_config(CONFIG_PATH)
codebook_config = config['parameter_codebook']
TRAIN_IMAGE_ROOT = PROJECT_ROOT / codebook_config['training_image_root']
TEST_IMAGE_ROOT = PROJECT_ROOT / config['demo']['image_root']
MODEL_DIR = PROJECT_ROOT / 'models' / 'SwinJSCC'
ARTIFACT_PATH = PROJECT_ROOT / codebook_config['artifact_path']

training_images = discover_images(TRAIN_IMAGE_ROOT)
test_images = discover_images(TEST_IMAGE_ROOT)
if len(training_images) < int(codebook_config['minimum_distinct_images']):
    raise RuntimeError(
        f"Need at least {codebook_config['minimum_distinct_images']} distinct local training images; found {len(training_images)}."
    )
training_hashes = [file_sha256(path) for path in training_images]
test_hashes = {file_sha256(path) for path in test_images}
if len(set(training_hashes)) != len(training_hashes):
    raise RuntimeError('Training image SHA-256 hashes are not unique.')
overlap = sorted(set(training_hashes).intersection(test_hashes))
if overlap:
    raise RuntimeError(f'Train/test image leakage detected by SHA-256: {overlap}')
if not MODEL_DIR.is_dir():
    raise FileNotFoundError(f'Missing manifest-validated Stage 1A model directory: {MODEL_DIR}')

RUN_CONTEXT.update(
    {
        'config': str(CONFIG_PATH),
        'training_image_root': str(TRAIN_IMAGE_ROOT),
        'training_image_count': len(training_images),
        'test_image_root': str(TEST_IMAGE_ROOT),
        'artifact_path': str(ARTIFACT_PATH),
        'demo_only': bool(codebook_config['demo_only']),
    }
)
print(json.dumps(RUN_CONTEXT, indent=2))


## 3. Load the offline Transformer teacher and SwinJSCC representation hook

In [ ]:
from TransformerPipeline.KnowledgeBases import NetworkXKnowledgeBase
from TransformerPipeline.clip_backbone import CLIPBackbone
from TransformerPipeline.pipeline import TransformerParameterPipeline
from TransformerPipeline.runtime import load_semantic_image_artifact, settings_from_config, task_context_from_mapping
from TransformerPipeline.observation_builder import parse_semantic_timestamp
from EncDecPipeline.Models.SwinJSCC.model_loader import load_stage1_swinjscc
from CodebookParam.swin_representation import capture_pre_modulation_representation

settings = settings_from_config(config)
task = task_context_from_mapping(config['demo']['task'])
reference_now = parse_semantic_timestamp(config['demo']['reference_now'])
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
clip_config = config['semantic']['clip']
teacher = TransformerParameterPipeline(
    settings=settings,
    embedding_backend=CLIPBackbone(
        model_name=clip_config['model_name'],
        device=device,
        local_files_only=bool(clip_config.get('local_files_only', False)),
    ),
    knowledge_base=NetworkXKnowledgeBase(),
)
adapter = load_stage1_swinjscc(MODEL_DIR, device=device)
encoder, decoder = adapter._require_built()
encoder.eval()
decoder.eval()
print({'device': device, 'teacher': teacher.name, 'task_profile_id': task.task_id})


## 4. Form offline Swin/CLIP teacher pairs

In [ ]:
from CodebookParam.semantic_codebook import CONTENT_PARAMETER_NAMES

representations = []
target_rows = []
for image_path, image_hash in zip(training_images, training_hashes, strict=True):
    bundle = teacher.extract_from_path(
        image_path,
        task,
        reference_now,
        captured_at=reference_now,
        metadata={'parameter_codebook_training': True},
    )
    targets = {}
    for parameter_name in CONTENT_PARAMETER_NAMES:
        parameter = bundle.get(parameter_name)
        if not parameter.available or parameter.value is None:
            raise RuntimeError(f'Offline teacher did not produce {parameter_name} for {image_path}.')
        targets[parameter_name] = parameter.value
    image_artifact = load_semantic_image_artifact(image_path)
    capture = capture_pre_modulation_representation(
        adapter,
        image_artifact,
        snr_db=float(config['semantic']['snr_db']),
        rate=int(config['semantic']['rate']),
    )
    representations.append(capture.representation.detach().cpu().numpy())
    target_rows.append(targets)

print({'paired_images': len(representations), 'target_fields': CONTENT_PARAMETER_NAMES})


## 5. Train and save the deterministic demo-only codebook

In [ ]:
from CodebookParam.semantic_codebook import SemanticParameterCodebook

semantic_codebook = SemanticParameterCodebook.train(
    representations,
    target_rows,
    task_profile_id=task.task_id,
    training_image_hashes=training_hashes,
    random_state=int(codebook_config['random_state']),
    codebook_size=int(codebook_config['codebook_size']),
    demo_only=bool(codebook_config['demo_only']),
    metadata={'run_context': RUN_CONTEXT},
)
saved_path = semantic_codebook.save(ARTIFACT_PATH)
summary_path = PROJECT_ROOT / 'results' / 'stage1c_parameter_codebook_demo_summary.json'
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(
    json.dumps(
        {
            **RUN_CONTEXT,
            'artifact_content_hash': semantic_codebook.content_hash(),
            'artifact_path': str(saved_path),
            'target_fields': CONTENT_PARAMETER_NAMES,
            'warning': 'demo_only: ten local images are an engineering smoke corpus, not a research training result',
        },
        indent=2,
        sort_keys=True,
    ),
    encoding='utf-8',
)
print({'artifact': str(saved_path), 'summary': str(summary_path), 'content_hash': semantic_codebook.content_hash()})


## 6. Interpretation

The saved artifact maps deterministic pooled Swin semantic features to three content-derived teacher targets: relevance, reliability, and task importance. AoI, freshness, urgency, and novelty remain runtime metadata/knowledge calculations. Do not report these ten-image results as model accuracy or generalization evidence.